# Passo 8 — Integração Wireshark: Análise de Tráfego de Rede

## 8.1 — Importações e Configuração

In [1]:
import pickle
import joblib
import re
import math
import pandas as pd
from urllib.parse import urlparse
from collections import Counter
from scapy.all import rdpcap, TCP, Raw

# Caminhos
PCAP_PATH   = '../captura.pcap'
MODELO_PATH = '../Aplicacao/modelo_rf_final.pkl'

print('Bibliotecas carregadas com sucesso.')

Bibliotecas carregadas com sucesso.


## 8.2 — Carregar o Modelo Random Forest

In [2]:
modelo = joblib.load(MODELO_PATH)

print(f'Modelo carregado: {type(modelo).__name__}')
print(f'Número de árvores: {modelo.n_estimators}')

Modelo carregado: RandomForestClassifier
Número de árvores: 700


## 8.3 — Função de Extração de Features (27 features lexicais)

In [3]:
ENCURTADORES = {
    'bit.ly', 'tinyurl.com', 'goo.gl', 't.co', 'ow.ly',
    'is.gd', 'buff.ly', 'adf.ly', 'short.link', 'rebrand.ly', 'tiny.cc'
}
TLDS_SUSPEITOS = {
    'tk', 'ml', 'ga', 'cf', 'gq', 'xyz', 'top', 'club', 'work',
    'info', 'biz', 'click', 'link', 'online', 'site', 'website'
}
PALAVRAS_SUSPEITAS = [
    'login', 'verify', 'secure', 'update', 'bank', 'paypal',
    'account', 'password', 'confirm', 'validate', 'signin',
    'wallet', 'billing', 'support', 'alert', 'security'
]

def calcular_entropia(texto):
    if not texto: return 0.0
    f = {}
    for c in texto:
        f[c] = f.get(c, 0) + 1
    e = 0.0
    for v in f.values():
        p = v / len(texto)
        e -= p * math.log2(p)
    return round(e, 4)

def extrair_features(url):
    url = str(url).strip()
    if url.startswith(('http://www.', 'https://www.')):
        url = url.replace('://www.', '://', 1)
    elif url.startswith('www.'):
        url = url[4:]

    dominio = ''; path = ''; query = ''; esquema = ''; porto = None
    try:
        if not url.startswith(('http://', 'https://')):
            parsed = urlparse('http://' + url)
        else:
            parsed = urlparse(url)
        dominio = parsed.netloc or parsed.path.split('/')[0]
        path    = parsed.path
        query   = parsed.query
        esquema = parsed.scheme
        try:
            porto = parsed.port
        except ValueError:
            porto = None
    except ValueError:
        pass

    dominio_limpo = dominio
    partes_dom    = dominio_limpo.split('.')
    tld           = partes_dom[-1].lower() if len(partes_dom) > 1 else ''
    url_lower     = url.lower()
    comp_url      = len(url)
    n_digitos     = sum(c.isdigit() for c in url)
    n_hifenes     = url.count('-')
    n_underscore  = url.count('_')
    n_percent     = url.count('%')
    n_iguais      = url.count('=')

    return {
        'lex_comp_url'            : comp_url,
        'lex_comp_dominio'        : len(dominio_limpo),
        'lex_comp_path'           : len(path),
        'lex_n_pontos'            : url.count('.'),
        'lex_n_hifenes'           : n_hifenes,
        'lex_n_subdominios'       : max(0, len(dominio_limpo.split('.')) - 2),
        'lex_n_digitos'           : n_digitos,
        'lex_n_barras'            : url.count('/'),
        'lex_n_iguais'            : n_iguais,
        'lex_n_ampersands'        : url.count('&'),
        'lex_n_percent'           : n_percent,
        'lex_n_underscores'       : n_underscore,
        'lex_n_params'            : len([p for p in query.split('&') if p]),
        'lex_ratio_digitos'       : round(n_digitos / comp_url, 4) if comp_url > 0 else 0,
        'lex_ratio_especiais'     : round((n_hifenes + n_underscore + n_percent + n_iguais) / comp_url, 4) if comp_url > 0 else 0,
        'lex_profundidade'        : len([p for p in path.split('/') if p]),
        'lex_tem_https'           : 1 if esquema == 'https' else 0,
        'lex_tem_ip'              : 1 if re.match(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', dominio) else 0,
        'lex_tem_arroba'          : 1 if '@' in url else 0,
        'lex_tem_http_no_path'    : 1 if 'http' in path.lower() else 0,
        'lex_tld_suspeito'        : 1 if tld in TLDS_SUSPEITOS else 0,
        'lex_encurtador'          : 1 if any(e in url_lower for e in ENCURTADORES) else 0,
        'lex_porto_suspeito'      : 1 if (porto and porto not in {80, 443, 8080}) else 0,
        'lex_digitos_consecutivos': 1 if re.search(r'\d{3,}', dominio_limpo) else 0,
        'lex_palavras_suspeitas'  : sum(1 for p in PALAVRAS_SUSPEITAS if p in url_lower),
        'lex_entropia_dominio'    : calcular_entropia(dominio_limpo),
        'lex_entropia_url'        : calcular_entropia(url),
    }

FEATURE_ORDER = [
    'lex_comp_url', 'lex_comp_dominio', 'lex_comp_path',
    'lex_n_pontos', 'lex_n_hifenes', 'lex_n_subdominios',
    'lex_n_digitos', 'lex_n_barras', 'lex_n_iguais',
    'lex_n_ampersands', 'lex_n_percent', 'lex_n_underscores',
    'lex_n_params', 'lex_ratio_digitos', 'lex_ratio_especiais',
    'lex_profundidade', 'lex_tem_https', 'lex_tem_ip',
    'lex_tem_arroba', 'lex_tem_http_no_path', 'lex_tld_suspeito',
    'lex_encurtador', 'lex_porto_suspeito', 'lex_digitos_consecutivos',
    'lex_palavras_suspeitas', 'lex_entropia_dominio', 'lex_entropia_url',
]

print('Função extrair_features() definida com 27 features.')

Função extrair_features() definida com 27 features.


## 8.4 — Extração de URLs do ficheiro .pcap

In [4]:
def extrair_urls_pcap(pcap_path):
    """Lê um ficheiro .pcap com scapy e extrai URLs HTTP."""
    urls_encontrados = []
    
    print(f'A ler ficheiro: {pcap_path}')
    pacotes = rdpcap(pcap_path)
    print(f'Total de pacotes lidos: {len(pacotes)}')
    
    for pacote in pacotes:
        if not (pacote.haslayer(TCP) and pacote.haslayer(Raw)):
            continue
        try:
            payload = pacote[Raw].load.decode('utf-8', errors='ignore')
        except Exception:
            continue
        if not payload.startswith(('GET ', 'POST ', 'HEAD ')):
            continue
        linhas = payload.split('\r\n')
        uri    = linhas[0].split(' ')[1] if len(linhas[0].split(' ')) > 1 else '/'
        host   = ''
        for linha in linhas[1:]:
            if linha.lower().startswith('host:'):
                host = linha.split(':', 1)[1].strip()
                break
        if host:
            url = f'http://{host}{uri}'
            urls_encontrados.append(url)
    
    urls_unicos = list(dict.fromkeys(urls_encontrados))
    print(f'Total de pedidos HTTP capturados : {len(urls_encontrados)}')
    print(f'URLs únicos (sem duplicados)     : {len(urls_unicos)}')
    return urls_unicos

urls_capturados = extrair_urls_pcap(PCAP_PATH)

A ler ficheiro: ../captura.pcap
Total de pacotes lidos: 11158
Total de pedidos HTTP capturados : 203
URLs únicos (sem duplicados)     : 198


## 8.5 — Visualizar os URLs Extraídos

In [5]:
print(f'\nPrimeiros 10 URLs extraídos do tráfego de rede:\n')
for i, url in enumerate(urls_capturados[:10], 1):
    print(f'  {i:2d}. {url}')

if not urls_capturados:
    print('AVISO: Nenhum URL HTTP encontrado na captura.')


Primeiros 10 URLs extraídos do tráfego de rede:

   1. http://streamhd4k.com/
   2. http://ww38.bobmovies.us/
   3. http://ww38.bobmovies.us/favicon.ico
   4. http://streamhd4k.com/_suggest?t=Best+4K+Streaming+Services&d=streamhd4k.com&s=8230795072983269
   5. http://streamhd4k.com/_suggest?t=Watch+Movies+Online+HD&d=streamhd4k.com&dest=https%3A%2F%2Fanswerharbor.com%2F2026%2F03%2F03%2Fwatch-movies-online-hd-instantly%2F%3Ffi%3D0%26cid%3D3c4ac6a6-e084-40ba-8d49-57498b22786e%26rac%3Dstreamhd4k.com%26sub%3Dstreamhd4k.com%26utm_source%3Dstreamhd4k.com%26hide_featured%3D1&s=8230795072983269
   6. http://www.china.com.cn/
   7. http://images.china.cn/images1/resource/styles/reset-1.0.css
   8. http://images.china.cn/images1/ch/2022lianghui/swiper-bundle.min.css
   9. http://images.china.cn/images1/ch/2015china/phone2023.js
  10. http://images.china.cn/images1/ch/2022ChinaIdex/css/china_footer.min.css


## 8.6 — Classificação dos URLs com o Modelo Random Forest

In [6]:
def classificar_urls(urls, modelo, feature_order):
    if not urls:
        print('Sem URLs para classificar.')
        return pd.DataFrame()
    resultados = []
    for url in urls:
        features      = extrair_features(url)
        X             = pd.DataFrame([features])[feature_order]
        predicao      = modelo.predict(X)[0]
        probabilidade = modelo.predict_proba(X)[0]
        prob_maligno  = round(probabilidade[1] * 100, 2)
        prob_benigno  = round(probabilidade[0] * 100, 2)
        resultados.append({
            'url'           : url,
            'classificacao' : 'MALICIOSO' if predicao == 1 else 'BENIGNO',
            'prob_benigno'  : prob_benigno,
            'prob_malicioso': prob_maligno,
            'confianca'     : max(prob_benigno, prob_maligno)
        })
    return pd.DataFrame(resultados)

df_resultados = classificar_urls(urls_capturados, modelo, FEATURE_ORDER)

if not df_resultados.empty:
    print(f'URLs classificados : {len(df_resultados)}')
    print(f"Benignos           : {(df_resultados['classificacao'] == 'BENIGNO').sum()}")
    print(f"Maliciosos         : {(df_resultados['classificacao'] == 'MALICIOSO').sum()}")

URLs classificados : 198
Benignos           : 151
Maliciosos         : 47


## 8.7 — Resultados Detalhados

In [7]:
if not df_resultados.empty:
    pd.set_option('display.max_colwidth', 80)
    pd.set_option('display.float_format', '{:.2f}'.format)
    print('=== TODOS OS URLs CLASSIFICADOS ===\n')
    print(df_resultados[['url', 'classificacao', 'prob_benigno', 'prob_malicioso', 'confianca']].to_string(index=False))
    maliciosos = df_resultados[df_resultados['classificacao'] == 'MALICIOSO']
    if not maliciosos.empty:
        print(f'\n=== URLs MALICIOSOS DETETADOS ({len(maliciosos)}) ===\n')
        print(maliciosos[['url', 'prob_malicioso', 'confianca']].to_string(index=False))
    else:
        print('\nNenhum URL malicioso detetado na captura.')

=== TODOS OS URLs CLASSIFICADOS ===

                                                                                                                                                                                                                                                                                                                                         url classificacao  prob_benigno  prob_malicioso  confianca
                                                                                                                                                                                                                                                                                                                      http://streamhd4k.com/       BENIGNO         74.95           25.05      74.95
                                                                                                                                                                                           

## 8.8 — Estatísticas da Análise

In [8]:
if not df_resultados.empty:
    total        = len(df_resultados)
    n_benignos   = (df_resultados['classificacao'] == 'BENIGNO').sum()
    n_maliciosos = (df_resultados['classificacao'] == 'MALICIOSO').sum()

    print('=== SUMÁRIO DA ANÁLISE DE TRÁFEGO DE REDE ===\n')
    print(f'Total de URLs analisados  : {total}')
    print(f'URLs Benignos             : {n_benignos} ({n_benignos/total*100:.1f}%)')
    print(f'URLs Maliciosos           : {n_maliciosos} ({n_maliciosos/total*100:.1f}%)')
    print(f'Confiança média (geral)   : {df_resultados["confianca"].mean():.2f}%')

    print('\n=== TOP 10 DOMÍNIOS MAIS FREQUENTES ===\n')
    dominios    = df_resultados['url'].apply(lambda u: urlparse(u).netloc)
    top_dominios = Counter(dominios).most_common(10)
    for dominio, contagem in top_dominios:
        print(f'  {dominio:<40} {contagem} pedido(s)')

=== SUMÁRIO DA ANÁLISE DE TRÁFEGO DE REDE ===

Total de URLs analisados  : 198
URLs Benignos             : 151 (76.3%)
URLs Maliciosos           : 47 (23.7%)
Confiança média (geral)   : 61.46%

=== TOP 10 DOMÍNIOS MAIS FREQUENTES ===

  images.china.cn                          119 pedido(s)
  vas.vtiger.in                            32 pedido(s)
  sneaindia.com                            25 pedido(s)
  www.faqs.org                             8 pedido(s)
  streamhd4k.com                           3 pedido(s)
  ww38.bobmovies.us                        2 pedido(s)
  www.china.com.cn                         2 pedido(s)
  www.360doc.com                           2 pedido(s)
  128.199.120.34                           2 pedido(s)
  xinhuanet.com                            1 pedido(s)


## 8.9 — Relatório HTML dos Resultados

In [9]:
def gerar_relatorio_html(df, output_path='../relatorio_wireshark.html'):
    total        = len(df)
    n_benignos   = (df['classificacao'] == 'BENIGNO').sum()
    n_maliciosos = (df['classificacao'] == 'MALICIOSO').sum()
    conf_media   = df['confianca'].mean()

    def linha_tabela(row):
        cor = '#ffecec' if row['classificacao'] == 'MALICIOSO' else '#ecffec'
        badge = (
            '<span style="background:#e74c3c;color:white;padding:2px 8px;border-radius:4px;font-size:12px">MALICIOSO</span>'
            if row['classificacao'] == 'MALICIOSO'
            else '<span style="background:#27ae60;color:white;padding:2px 8px;border-radius:4px;font-size:12px">BENIGNO</span>'
        )
        return f"""
        <tr style="background:{cor}">
            <td style="word-break:break-all;max-width:500px">{row['url']}</td>
            <td style="text-align:center">{badge}</td>
            <td style="text-align:center">{row['prob_benigno']:.1f}%</td>
            <td style="text-align:center">{row['prob_malicioso']:.1f}%</td>
            <td style="text-align:center">{row['confianca']:.1f}%</td>
        </tr>"""

    linhas_html = ''.join(linha_tabela(row) for _, row in df.iterrows())

    html = f"""<!DOCTYPE html>
<html lang="pt">
<head>
    <meta charset="UTF-8">
    <title>Relatório Wireshark — URL Shield</title>
    <style>
        body {{ font-family: Segoe UI, Arial, sans-serif; margin: 40px; color: #222; }}
        h1 {{ color: #2c3e50; border-bottom: 3px solid #2980b9; padding-bottom: 10px; }}
        h2 {{ color: #2980b9; margin-top: 30px; }}
        .cards {{ display: flex; gap: 20px; margin: 20px 0; }}
        .card {{ flex: 1; padding: 20px; border-radius: 8px; text-align: center; color: white; }}
        .card-total {{ background: #2980b9; }}
        .card-benigno {{ background: #27ae60; }}
        .card-malicioso {{ background: #e74c3c; }}
        .card-confianca {{ background: #8e44ad; }}
        .card h3 {{ margin: 0; font-size: 2em; }}
        .card p {{ margin: 5px 0 0; font-size: 0.9em; opacity: 0.9; }}
        table {{ width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 13px; }}
        th {{ background: #2c3e50; color: white; padding: 10px; text-align: left; }}
        td {{ padding: 8px 10px; border-bottom: 1px solid #ddd; }}
        tr:hover {{ filter: brightness(0.96); }}
        .footer {{ margin-top: 40px; font-size: 12px; color: #888; text-align: center; }}
    </style>
</head>
<body>
    <h1>🔍 Relatório de Análise de Tráfego de Rede</h1>
    <p><strong>Projeto II — Deteção de URLs Maliciosos</strong> | Kyrylo Sakhnenko & Rodrigo Figueiredo | ESTCB, IPCB 2025/2026</p>

    <h2>Sumário</h2>
    <div class="cards">
        <div class="card card-total">
            <h3>{total}</h3><p>URLs Analisados</p>
        </div>
        <div class="card card-benigno">
            <h3>{n_benignos}</h3><p>Benignos ({n_benignos/total*100:.1f}%)</p>
        </div>
        <div class="card card-malicioso">
            <h3>{n_maliciosos}</h3><p>Maliciosos ({n_maliciosos/total*100:.1f}%)</p>
        </div>
        <div class="card card-confianca">
            <h3>{conf_media:.1f}%</h3><p>Confiança Média</p>
        </div>
    </div>

    <h2>Resultados Detalhados</h2>
    <table>
        <thead>
            <tr>
                <th>URL</th>
                <th style="text-align:center">Classificação</th>
                <th style="text-align:center">P(Benigno)</th>
                <th style="text-align:center">P(Malicioso)</th>
                <th style="text-align:center">Confiança</th>
            </tr>
        </thead>
        <tbody>
            {linhas_html}
        </tbody>
    </table>

    <div class="footer">
        Gerado automaticamente pelo pipeline de análise de tráfego — Projeto II, 2025/2026
    </div>
</body>
</html>"""

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html)
    print(f'Relatório HTML gerado: {output_path}')

if not df_resultados.empty:
    gerar_relatorio_html(df_resultados)

Relatório HTML gerado: ../relatorio_wireshark.html
